In [1]:
!pwd

/root/jupyter_notebooks/FYP


In [2]:
import numpy as np
from scipy import signal
import tensorflow as tf
from tensorflow.keras import layers, Model, initializers, Sequential
from optic.models.devices import mzm, photodiode, edfa, iqm, coherentReceiver, pdmCoherentReceiver, basicLaserModel
from optic.models.channels import linearFiberChannel, ssfm
from optic.comm.modulation import modulateGray, grayMapping
from optic.comm.sources import bitSource, symbolSource
from optic.dsp.core import upsample, pulseShape, pnorm, anorm, signalPower, firFilter, decimate, symbolSync,phaseNoise

try:
    from optic.dsp.coreGPU import checkGPU
    if checkGPU():
        from optic.dsp.coreGPU import firFilter
    else:
        from optic.dsp.core import firFilter
except ImportError:
    from optic.dsp.core import firFilter

from optic.utils import parameters, dBm2W, ber2Qfactor
from optic.plot import eyediagram, pconst, plotPSD
import matplotlib.pyplot as plt
from scipy.special import erfc
from tqdm.notebook import tqdm
import scipy as sp
import scipy.constants as const

try:
    from optic.models.modelsGPU import manakovSSF
except:
    from optic.models.channels import manakovSSF

from optic.dsp.equalization import edc, mimoAdaptEqualizer, ffe
from optic.dsp.carrierRecovery import cpr
from optic.comm.metrics import fastBERcalc, monteCarloGMI, monteCarloMI, calcEVM, bert
from optic.dsp.clockRecovery import gardnerClockRecovery


import logging as logg
logg.basicConfig(level=logg.INFO, format='%(message)s', force=True)
import time
from helper_funcs import *

In [3]:
from pynq import Overlay
from pynq import allocate
import numpy as np
overlay = Overlay('/root/jupyter_notebooks/FYP/vivado_export.xsa')

In [ ]:
from pynq import ps

print(ps.Clocks.fclk0_mhz)
ps.Clocks.fclk0_mhz = 250 # try different values. default (no violation) set at ~130mhz
print(ps.Clocks.fclk0_mhz)
print(ps.Clocks.cpu_mhz)

111.11
249.9975
1333.32


In [5]:
help(ps.Clocks)

Help on class Clocks in module pynq.ps:

class Clocks(builtins.object)
 |  Class for all the PS and PL clocks exposed to users.
 |  
 |  With this class, users can get the CPU clock and all the PL clocks. Users
 |  can also set PL clocks to other values using this class.
 |  
 |  Attributes
 |  ----------
 |  cpu_mhz : float
 |      The clock rate of the CPU, measured in MHz.
 |  fclk0_mhz : float
 |      The clock rate of the PL clock 0, measured in MHz.
 |  fclk1_mhz : float
 |      The clock rate of the PL clock 1, measured in MHz.
 |  fclk2_mhz : float
 |      The clock rate of the PL clock 2, measured in MHz.
 |  fclk3_mhz : float
 |      The clock rate of the PL clock 3, measured in MHz.
 |  
 |  Data descriptors defined here:
 |  
 |  __dict__
 |      dictionary for instance variables (if defined)
 |  
 |  __weakref__
 |      list of weak references to the object (if defined)



In [6]:
ip = overlay.NNDPD_0
mmio = ip.mmio
register_map = ip.register_map
registers = register_map._register_classes

In [7]:
for name, reg in registers.items():
    print(name, reg)

CTRL (<class 'pynq.registers.RegisterCTRL'>, 0, 32, None, None, 'read-write')
GIER (<class 'pynq.registers.RegisterGIER'>, 4, 32, None, None, 'read-write')
IP_IER (<class 'pynq.registers.RegisterIP_IER'>, 8, 32, None, None, 'read-write')
IP_ISR (<class 'pynq.registers.RegisterIP_ISR'>, 12, 32, None, None, 'read-write')
sigI_in_1 (<class 'pynq.registers.RegistersigI_in_1'>, 16, 32, None, None, 'write-only')
sigI_in_2 (<class 'pynq.registers.RegistersigI_in_2'>, 20, 32, None, None, 'write-only')
sigQ_in_1 (<class 'pynq.registers.RegistersigQ_in_1'>, 28, 32, None, None, 'write-only')
sigQ_in_2 (<class 'pynq.registers.RegistersigQ_in_2'>, 32, 32, None, None, 'write-only')
sigI_out_1 (<class 'pynq.registers.RegistersigI_out_1'>, 40, 32, None, None, 'write-only')
sigI_out_2 (<class 'pynq.registers.RegistersigI_out_2'>, 44, 32, None, None, 'write-only')
sigQ_out_1 (<class 'pynq.registers.RegistersigQ_out_1'>, 52, 32, None, None, 'write-only')
sigQ_out_2 (<class 'pynq.registers.RegistersigQ_ou

In [8]:
help(register_map.CTRL)

Help on RegisterCTRL in module pynq.registers object:

class RegisterCTRL(Register)
 |  RegisterCTRL(address, width=32, debug=False, buffer=None, access='read-write')
 |  
 |  Control signals
 |  
 |  Method resolution order:
 |      RegisterCTRL
 |      Register
 |      builtins.object
 |  
 |  Readonly properties defined here:
 |  
 |  AP_DONE
 |      Control signal Register for 'ap_done'.
 |  
 |  AP_IDLE
 |      Control signal Register for 'ap_idle'.
 |  
 |  AP_READY
 |      Control signal Register for 'ap_ready'.
 |  
 |  RESERVED_1
 |      Reserved.  0s on read.
 |  
 |  RESERVED_2
 |      Reserved.  0s on read.
 |  
 |  INTERRUPT
 |      Control signal Register for 'interrupt'.
 |  
 |  RESERVED_3
 |      Reserved.  0s on read.
 |  
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |  
 |  AP_START
 |      Control signal Register for 'ap_start'.
 |  
 |  AUTO_RESTART
 |      Control signal Register for 'auto_restart'.

In [9]:
# Allocated buffer (m_axi)
no_symbols = 5000
input_buffer_size = no_symbols
output_buffer_size = no_symbols

sigI_in_buffer = allocate(shape=(no_symbols,), dtype=np.float32)
sigQ_in_buffer = allocate(shape=(no_symbols,), dtype=np.float32)
sigI_out_buffer = allocate(shape=(no_symbols,), dtype=np.float32)
sigQ_out_buffer = allocate(shape=(no_symbols,), dtype=np.float32)


In [10]:
register_map.sigI_in_1.sigI_in = sigI_in_buffer.device_address # no need for the upper 32bits
register_map.sigQ_in_1.sigQ_in = sigQ_in_buffer.device_address # no need for the upper 32bits
register_map.sigI_out_1.sigI_out = sigI_out_buffer.device_address # no need for the upper 32bits
register_map.sigQ_out_1.sigQ_out = sigQ_out_buffer.device_address # no need for the upper 32bits


In [11]:
def build_model():
    inputs = layers.Input(shape=(None, 2)) # 2 for I and Q

    sec_a = layers.Conv1D(2, 101, padding='same')(inputs) # 100 taps was a sweet spot, 20-ish fails to converge, tiker with different values.

    nonlinear_1 = layers.Dense(20, activation=tf.math.sin)(sec_a)
    nonlinear_2 = layers.Dense(20, activation=tf.math.sin)(nonlinear_1)
    nonlinear_3 = layers.Dense(2, activation='linear')(nonlinear_2)
    
    outputs = layers.Add()([sec_a, nonlinear_3]) 
    
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2), loss='mse') # ILA)
    model.load_weights("best_model.weights.h5")
    return model


In [12]:
def float_to_q4_20(value):
    # 1. Configuration
    fractional_bits = 20
    total_bits = 24
    
    # 2. Scaling
    # Multiply by 2^20
    scaled_value = round(value * (1 << fractional_bits))
    
    # 3. Clamping (Range Handling)
    # Max: 2^(23) - 1, Min: -2^(23)
    max_val = (1 << (total_bits - 1)) - 1
    min_val = -(1 << (total_bits - 1))
    
    clamped_value = max(min(scaled_value, max_val), min_val)
    
    # 4. Convert to 24-bit unsigned representation (for hardware/storage)
    # This handles negative numbers using two's complement
    return clamped_value & 0xFFFFFF


def q4_20_to_float(value):
    """
    Converts a 24-bit fixed-point (Q4.20) value back to a float.
    Handles a single integer or a NumPy array.
    """
    fractional_bits = 20
    
    # 1. Sign Extension (if bit 23 is set, it's negative)
    # We use (value ^ 0x800000) - 0x800000 for a fast bitwise sign extension
    if isinstance(value, np.ndarray):
        # Vectorized sign extension for NumPy
        sign_bit = 1 << 23
        mask = (1 << 24) - 1
        # Convert unsigned 24-bit to signed integer
        signed_val = ((value & mask) ^ sign_bit) - sign_bit
    else:
        # Standard Python scalar logic
        sign_bit = 1 << 23
        signed_val = (value & 0xFFFFFF)
        if signed_val & sign_bit:
            signed_val -= (1 << 24)

    # 2. Scaling back to float
    return signed_val / (1 << fractional_bits)


Vq4_20_to_float = np.vectorize(q4_20_to_float)
Vfloat_to_q4_20 = np.vectorize(float_to_q4_20)

In [13]:
# Hardware accelerated function
def dpd_hw(dpd_input):
    symbTx_r = dpd_input[:, 0]
    symbTx_im = dpd_input[:, 1]
    # Write to input buffer
    sigI_in_buffer[:len(symbTx_r)] = symbTx_r
    sigQ_in_buffer[:len(symbTx_r)] = symbTx_im

    # Send start signal
    register_map.CTRL.AP_START = 1
    
    # Wait until algorithm has completed
    while (register_map.CTRL.AP_IDLE != 1):
        ...

In [14]:
M = 16
no_symbols= 100_000 # must be multiple of 5000 (seq_length)
nBits = int(no_symbols * np.log2(M))
SpSout = 2
mzmScale = 0.8
laserLinewidth = 100e3
dpd_model = build_model()

paramSymb = intialise_paramSymb(M, nBits, seed=333)
symbTx = symbolSource(paramSymb)

symbTx_nn = preprocess(symbTx) # shape = 20,5000,2

# dpd_model.predict()
single_batch = symbTx_nn[0][np.newaxis, ...] # shape = 1,5000,2

sw_time = %timeit -r 10 -o dpd_model.predict(single_batch, verbose=0)
hw_time = %timeit -r 10 -o dpd_hw(single_batch[0])

print(f"AVG Execution Time - CPU: {sw_time.average}")
print(f"AVG Execution Time - w/ FPGA Acceleration: {hw_time.average}")
print('Performance gain:', sw_time.average / hw_time.average)


/usr/local/share/pynq-venv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


311 ms ± 1.84 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
251 µs ± 247 ns per loop (mean ± std. dev. of 10 runs, 1,000 loops each)
AVG Execution Time - CPU: 0.3111151989000064
AVG Execution Time - w/ FPGA Acceleration: 0.0002511387366000008
Performance gain: 1238.8180458020404


In [ ]:
#prepping symbDPD for export to host for validation (optical sim faster there)
symbDPD_hw =[]
for batch in symbTx_nn:
    dpd_hw(batch)
    symbDPD_hw.append(sigI_out_buffer + 1j*sigQ_out_buffer)


symbDPD_sw = merge_i_q(dpd_model.predict(symbTx_nn)).flatten()
symbDPD_hw = np.array(symbDPD_hw).flatten()



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 285ms/step


In [23]:
symbDPD_hw

array([-0.79112625-0.26858234j, -0.86713123-5.1061087j ,
       -0.34039593-0.21674156j, ...,  0.2991867 -0.9022732j ,
       -0.91757584-0.30349445j, -0.8734188 -0.31493092j], dtype=complex64)

In [24]:
np.save("symbDPD_hw_data", symbDPD_hw)